# 01 – Dataförberedelse: Student Depression Dataset

Mål: gå från rådata till ett **rent, analysklart dataset** som `02_EDA.ipynb` och senare modellering kan bygga vidare på.

Steg: droppa irrelevanta kolumner, hantera saknade värden, städa smutsig data, koda om kategoriska variabler, spara resultatet.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

## 1. Ladda in rådata

In [2]:
df = pd.read_csv("Student Depression Dataset.csv")
print("Ursprunglig shape:", df.shape)
df.head()

Ursprunglig shape: (27901, 18)


,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,M.Tech,Yes,1.0,1.0,No,0


## 2. Droppa irrelevanta kolumner

`id` är bara ett radnummer utan prediktivt värde. `Profession`, `Work Pressure`, `Job Satisfaction` och `Degree`
hör i praktiken ihop med yrkesliv snarare än studentliv - majoriteten i datasetet är heltidsstudenter,
så dessa kolumner är mest tomma/irrelevanta för vårt problem (bekräftat i EDA-utforskningen).

In [3]:
cols_to_drop = ["id", "Profession", "Work Pressure", "Job Satisfaction", "Degree"]
df = df.drop(columns=cols_to_drop)
print("Shape efter kolumn-drop:", df.shape)

Shape efter kolumn-drop: (27901, 13)


## 3. Hantera saknade värden

In [4]:
missing = df.isnull().sum()
missing[missing > 0]

Financial Stress    3
dtype: int64

In [5]:
# Endast ett fåtal rader saknar Financial Stress (<0.1% av datasetet).
# Med så få rader är det säkrare att droppa dem än att gissa ett imputerat värde
# som kan snedvrida sambandet med Depression.
before = len(df)
df = df.dropna(subset=["Financial Stress"])
print(f"Droppade {before - len(df)} rader p.g.a. saknad Financial Stress")

Droppade 3 rader p.g.a. saknad Financial Stress


## 4. Städa City-kolumnen

En snabb koll visar att `City` innehåller felinmatad data - poster som `"Saanvi"`, `"M.Tech"`, `"3.0"`,
`"Less than 5 Kalyan"` - troligen data som råkat hamna i fel kolumn vid insamlingen. Dessa har alla
väldigt få förekomster (1-2 st) jämfört med riktiga städer (400+ st vardera), vilket gör att vi
kan skilja ut dem med ett enkelt antal-tröskelvärde.

In [6]:
city_counts = df["City"].value_counts()
print("Minsta antal bland de 'riktiga' städerna:", city_counts[city_counts > 50].min())
print("Antal misstänkta felposter (<= 50 förekomster):", (city_counts <= 50).sum())
city_counts[city_counts <= 50]

Minsta antal bland de 'riktiga' städerna: 461
Antal misstänkta felposter (<= 50 förekomster): 22


City
Saanvi                2
Bhavna                2
City                  2
Harsha                2
M.Tech                1
Less Delhi            1
3.0                   1
Less than 5 Kalyan    1
Mira                  1
Vaanya                1
Gaurav                1
Harsh                 1
Reyansh               1
Kibara                1
Rashi                 1
ME                    1
M.Com                 1
Nalyan                1
Mihir                 1
Nalini                1
Nandini               1
Khaziabad             1
Name: count, dtype: int64

In [7]:
valid_cities = city_counts[city_counts > 50].index
before = len(df)
df = df[df["City"].isin(valid_cities)]
print(f"Droppade {before - len(df)} rader med ogiltig/felinmatad City")

Droppade 26 rader med ogiltig/felinmatad City


## 5. Hantera oklara kategorier ("Others")

`Sleep Duration` och `Dietary Habits` har båda en liten `"Others"`-kategori (18 respektive 12 rader).
Vi vet inte vad "Others" faktiskt betyder, och med så få rader väger det inte tillräckligt tungt
för att motivera en gissning - vi droppar dem istället för att undvika att gissa fel.

In [8]:
before = len(df)
df = df[(df["Sleep Duration"] != "Others") & (df["Dietary Habits"] != "Others")]
print(f"Droppade {before - len(df)} rader med oklar Sleep Duration/Dietary Habits")

Droppade 30 rader med oklar Sleep Duration/Dietary Habits


## 6. Koda om kategoriska variabler

- **Sleep Duration** och **Dietary Habits** har en naturlig ordning (mindre -> mer sömn, sämre -> bättre kost),
  så vi ordinal-kodar dem (1, 2, 3, ...) istället för one-hot - det bevarar ordningen för modellerna.
- **Gender**, **Have you ever had suicidal thoughts ?** och **Family History of Mental Illness** är binära
  (Ja/Nej-typ) och kodas som 0/1.
- **City** lämnas som text tills vidare - 32 giltiga städer är för många för enkel one-hot-kodning här;
  vi tar beslut om kodningsstrategi (t.ex. target- eller frekvenskodning) i modelleringssteget.

In [9]:
sleep_map = {
    "Less than 5 hours": 1,
    "5-6 hours": 2,
    "7-8 hours": 3,
    "More than 8 hours": 4,
}
diet_map = {
    "Unhealthy": 1,
    "Moderate": 2,
    "Healthy": 3,
}
binary_map = {"Yes": 1, "No": 0}
gender_map = {"Female": 0, "Male": 1}

df["Sleep Duration"] = df["Sleep Duration"].map(sleep_map)
df["Dietary Habits"] = df["Dietary Habits"].map(diet_map)
df["Have you ever had suicidal thoughts ?"] = df["Have you ever had suicidal thoughts ?"].map(binary_map)
df["Family History of Mental Illness"] = df["Family History of Mental Illness"].map(binary_map)
df["Gender"] = df["Gender"].map(gender_map)

# Sparar kodningsnycklarna som en liten referenstabell - använd denna för att tolka
# siffrorna i 02_EDA.ipynb och i eventuell rapport.
encoding_key = {
    "Sleep Duration": sleep_map,
    "Dietary Habits": diet_map,
    "Have you ever had suicidal thoughts ? / Family History of Mental Illness": binary_map,
    "Gender": gender_map,
}
for col, mapping in encoding_key.items():
    print(f"{col}: {mapping}")

Sleep Duration: {'Less than 5 hours': 1, '5-6 hours': 2, '7-8 hours': 3, 'More than 8 hours': 4}
Dietary Habits: {'Unhealthy': 1, 'Moderate': 2, 'Healthy': 3}
Have you ever had suicidal thoughts ? / Family History of Mental Illness: {'Yes': 1, 'No': 0}
Gender: {'Female': 0, 'Male': 1}


## 7. Kontroll av resultat

In [10]:
print("Slutlig shape:", df.shape)
df.info()

Slutlig shape: (27842, 13)
<class 'pandas.DataFrame'>
Index: 27842 entries, 0 to 27900
Data columns (total 13 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Gender                                 27842 non-null  int64  
 1   Age                                    27842 non-null  float64
 2   City                                   27842 non-null  str    
 3   Academic Pressure                      27842 non-null  float64
 4   CGPA                                   27842 non-null  float64
 5   Study Satisfaction                     27842 non-null  float64
 6   Sleep Duration                         27842 non-null  int64  
 7   Dietary Habits                         27842 non-null  int64  
 8   Have you ever had suicidal thoughts ?  27842 non-null  int64  
 9   Work/Study Hours                       27842 non-null  float64
 10  Financial Stress                       27842 non-null  floa

In [11]:
# Sanity check: inga saknade värden kvar
assert df.isnull().sum().sum() == 0, "Det finns fortfarande saknade värden!"

# Målvariabelns balans efter städning - bör vara ungefär samma som innan (58.5/41.5)
df["Depression"].value_counts(normalize=True)

Depression
1    0.585554
0    0.414446
Name: proportion, dtype: float64

In [12]:
df.head()

,Gender,Age,City,Academic Pressure,CGPA,Study Satisfaction,Sleep Duration,Dietary Habits,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,1,33.0,Visakhapatnam,5.0,8.97,2.0,2,3,1,3.0,1.0,0,1
1,0,24.0,Bangalore,2.0,5.90,5.0,2,2,0,3.0,2.0,1,0
2,1,31.0,Srinagar,3.0,7.03,5.0,1,3,0,9.0,1.0,1,0
3,0,28.0,Varanasi,3.0,5.59,2.0,3,2,1,4.0,5.0,1,1
4,0,25.0,Jaipur,4.0,8.13,3.0,2,2,1,1.0,1.0,0,0


## 8. Spara rent dataset

In [13]:
df.to_csv("student_depression_clean.csv", index=False)
print("Sparad som student_depression_clean.csv")

Sparad som student_depression_clean.csv
